# KioGuard FP16 공통 모델 비교 v1

이 노트북은 새 14개 의도와 출력 계약 v3으로 7개 모델을 같은 조건에서 비교합니다.

- 한 런타임에는 모델 하나만 로드합니다.
- 기본 `RUN_SCOPE="gate"`는 20건 호환성 확인입니다.
- 정식 후보 비교는 `RUN_SCOPE="selection"`으로 118건을 실행합니다.
- 기존 48회귀까지 포함하려면 `RUN_SCOPE="all"`을 사용합니다.
- 평가 데이터는 개발용 초안입니다. 사람 검수 전 결과는 최종 성능이 아닙니다.
- 과거 8개 라벨 점수와 이 결과를 합산하지 않습니다.
- llama.cpp/GGUF/PTQ는 FP16 후보를 1~2개로 줄인 다음 진행합니다.

현재 검증된 Colab 환경을 사용합니다. 패키지를 임의로 업그레이드하지 마세요. 아래 검증 셀이 실패하면 실행을 중지하고 torch/torchvision/torchaudio CUDA 버전을 먼저 맞춥니다.


In [ ]:
# 셀 1: Google Drive 연결과 실행 설정
from google.colab import drive
drive.mount("/content/drive")

MODEL_KEY = "qwen35_2b"  # 7개 중 하나. configs/model_matrix.json 참고
RUN_SCOPE = "gate"       # gate | selection | all
MAX_NEW_TOKENS = 220
SEED = 20260928

REPO_URL = "https://github.com/cjuproject34/Kioguard.git"
REPO_DIR = "/content/Kioguard"
DRIVE_OUTPUT_ROOT = "/content/drive/MyDrive/KioGuard/experiments/FP16_COMMON_V3"

print({"MODEL_KEY": MODEL_KEY, "RUN_SCOPE": RUN_SCOPE, "MAX_NEW_TOKENS": MAX_NEW_TOKENS})


In [ ]:
# 셀 2: 저장소와 평가 자료 확보
import os, subprocess

if not os.path.exists(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

EVAL_ROOT = os.path.join(REPO_DIR, "research", "llm-intent", "evaluation")
assert os.path.exists(os.path.join(EVAL_ROOT, "dataset_manifest.json")), "평가 자료가 저장소에 없습니다."
print("EVAL_ROOT:", EVAL_ROOT)


In [ ]:
# 셀 3: 런타임 검증. 설치 셀이 아니라 현재 환경을 확인하는 셀입니다.
import importlib.metadata as mdlib
import torch, torchvision, torchaudio, transformers
from transformers import AutoModelForCausalLM, AutoModelForMultimodalLM, AutoProcessor, AutoTokenizer

versions = {
    "torch": torch.__version__,
    "torch_cuda": torch.version.cuda,
    "torchvision": torchvision.__version__,
    "torchaudio": torchaudio.__version__,
    "transformers": transformers.__version__,
    "transformers_commit": getattr(transformers, "__commit__", None),
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
print(versions)
assert torch.cuda.is_available(), "GPU 런타임이 아닙니다."
assert "+cu128" in torch.__version__, "현재 검증 기준은 torch cu128입니다. 환경을 다시 확인하세요."
assert "+cu128" in torchvision.__version__, "torchvision CUDA 버전 불일치"
assert "+cu128" in torchaudio.__version__, "torchaudio CUDA 버전 불일치"
print("모든 import 성공")


In [ ]:
# 셀 4: 모델 행렬과 평가 데이터 로드
import json, pathlib

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

with open(os.path.join(EVAL_ROOT, "configs", "model_matrix.json"), encoding="utf-8") as f:
    matrix = json.load(f)
model_cfg = next(m for m in matrix["models"] if m["key"] == MODEL_KEY)

core_path = os.path.join(EVAL_ROOT, "data", "selection_dev_v0.1.jsonl")
challenge_path = os.path.join(EVAL_ROOT, "data", "selection_challenge_v0.1.jsonl")
legacy_path = os.path.join(EVAL_ROOT, "data", "legacy48_regression_v1.2.jsonl")
core = read_jsonl(core_path)
challenge = read_jsonl(challenge_path)
legacy = read_jsonl(legacy_path)

gate_ids = set()
for intent in matrix.get("intent_order", []):
    pass
for intent in ["SELECT", "MODIFY", "FULFILLMENT", "PAYMENT_METHOD", "RECEIPT", "BENEFIT_CONTROL", "NAVIGATE", "CANCEL", "GUIDE", "INFO", "UI_CONTROL", "HELP", "COMMIT_REQUEST", "RESET"]:
    gate_ids.add(next(x["case_id"] for x in core if any(r["intent"] == intent for r in x["gold"]["requests"])))
gate_ids.update(["CH-CLR-01", "CH-CLR-02", "CH-NEG-01", "CH-NEG-02", "CH-MUL-01", "CH-CTX-04"])

if RUN_SCOPE == "gate":
    cases = [x for x in core + challenge if x["case_id"] in gate_ids]
elif RUN_SCOPE == "selection":
    cases = core + challenge
elif RUN_SCOPE == "all":
    cases = core + challenge + legacy
else:
    raise ValueError("RUN_SCOPE must be gate, selection, or all")

print("model:", model_cfg)
print("cases:", len(cases), "core/challenge/legacy:", len(core), len(challenge), len(legacy))


In [ ]:
# 셀 5: prompt v3 고정
import hashlib

INTENT_RULES = """
SELECT: 새 상품·서류·서비스 선택
MODIFY: 선택 대상의 속성 설정·변경·교체
FULFILLMENT: 매장·포장·수령 방식 선택
PAYMENT_METHOD: 결제수단 선택, 거래 최종 확정은 아님
RECEIPT: 영수증 수령·거절·재발급
BENEFIT_CONTROL: 쿠폰·할인·적립 적용·해제·거절
NAVIGATE: 화면·단계 이동, 내용 초기화는 아님
CANCEL: 주문 항목·주문·신청·거래 취소
GUIDE: 수행 방법·버튼 위치·동선 안내
INFO: 지원·조건·가격·상태 등 사실 조회
UI_CONTROL: 글자·음량·속도·기존 안내 재생 제어
HELP: 일반 도움·직원 도움·장치/진행 문제 지원
COMMIT_REQUEST: 주문·결제·신청의 최종 확정·제출
RESET: 내용을 버리고 과업을 처음부터 재시작
""".strip()

FALLBACK_RULES = """
ASK_SCREEN_TARGET, ASK_GUIDANCE_NEED, ASK_TARGET, ASK_CONTEXT,
ASK_REFERENCE, ASK_INPUT_REPEAT, ASK_SCOPE, OUT_OF_SCOPE, NO_ACTION
""".replace("\n", " ").strip()

CANONICAL_RULES = """
새 주문 target은 NEW_ORDER, 새 신청은 NEW_APPLICATION입니다.
현재 주문은 CURRENT_ORDER, 현재 거래는 CURRENT_TRANSACTION, 앱 화면은 APP_UI입니다.
context에 item_id/application_id/transaction_id가 있으면 그 ID를 target으로 사용합니다.
온도는 WARM 또는 COLD, 이용 방식은 DINE_IN 또는 TAKEOUT입니다.
결제수단은 CARD, CASH, SAMSUNG_PAY입니다.
영수증 action은 RECEIVE, DECLINE, REISSUE입니다.
혜택 action은 APPLY, REMOVE, PARTICIPATE, DECLINE입니다.
화면 목적지는 BACK, HOME, PAYMENT, NEXT, DOCUMENT_LIST입니다.
UI action은 FONT_SIZE_UP, FONT_SIZE_DOWN, VOLUME_UP, SPEECH_RATE_DOWN, REPLAY, STOP_PLAYBACK입니다.
HELP의 help_type은 GENERAL_SUPPORT, HUMAN_ASSIST, ISSUE_SUPPORT입니다.
최종 확정 action은 COMMIT 또는 SUBMIT입니다.
정확한 값이 context에도 utterance에도 없으면 값을 만들지 말고 fallback을 사용합니다.
""".strip()

PROMPT_PREFIX = f"""당신은 KioGuard 키오스크 발화 의미 분석기입니다.
앱이 확인한 context와 사용자 utterance만 사용하세요.
context의 available_actions는 가능한 동작 목록이며 사용자 요청이 아닙니다.
사용자가 말하지 않은 취소, 확정, 초기화, 이동을 추가하지 마세요.
조건이 미확인이면 condition을 보존하고 필요한 fallback을 출력하세요.
복합 요청은 발화 순서대로 requests에 넣고 선행 관계는 depends_on으로 기록하세요.

허용 intent:
{INTENT_RULES}

허용 fallback type:
{FALLBACK_RULES}

정규화 규칙:
{CANONICAL_RULES}

반드시 Markdown 없이 JSON 객체 하나만 출력하세요.
최상위 키는 requests, fallback 두 개뿐입니다.
각 request 키는 request_id, intent, target, slots, condition, depends_on 여섯 개를 모두 포함합니다.
condition이 없으면 null, depends_on이 없으면 null, fallback이 없으면 null입니다.
request_id는 r1, r2 순서입니다.
fallback은 null 또는 {{"type":"허용값","field":"필요한 필드"}}입니다.
빈 요청의 유효한 형식은 {{"requests":[],"fallback":null}}입니다.
"""
PROMPT_HASH = hashlib.sha256(PROMPT_PREFIX.encode("utf-8")).hexdigest()

def build_prompt(case):
    payload = {
        "context": case["context"],
        "utterance": case["utterance"],
    }
    return PROMPT_PREFIX + "\n입력:\n" + json.dumps(payload, ensure_ascii=False, separators=(",", ":"))

print("prompt_sha256:", PROMPT_HASH)
print(build_prompt(cases[0])[:1500])


In [ ]:
# 셀 6: Hugging Face 인증 토큰을 안전하게 가져옵니다. 토큰은 출력하지 않습니다.
hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    pass

if model_cfg["model_id"].startswith("google/") and not hf_token:
    print("Gemma gated 모델이면 Colab Secrets에 HF_TOKEN을 등록해야 할 수 있습니다.")
else:
    print("인증 준비 완료")


In [ ]:
# 셀 7: 선택 모델 하나 로드
import gc, os, psutil, time

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
process = psutil.Process(os.getpid())
rss_before = process.memory_info().rss / 1024**3
load_start = time.perf_counter()

common_kwargs = {
    "revision": model_cfg["revision"],
    "torch_dtype": torch.float16,
    "device_map": "auto",
    "low_cpu_mem_usage": True,
}
if hf_token:
    common_kwargs["token"] = hf_token

if model_cfg["loader"] == "multimodal":
    processor = AutoProcessor.from_pretrained(model_cfg["model_id"], revision=model_cfg["revision"], token=hf_token)
    model = AutoModelForMultimodalLM.from_pretrained(model_cfg["model_id"], **common_kwargs)
    tokenizer = None
else:
    tokenizer = AutoTokenizer.from_pretrained(model_cfg["model_id"], revision=model_cfg["revision"], token=hf_token)
    processor = None
    model = AutoModelForCausalLM.from_pretrained(model_cfg["model_id"], **common_kwargs)

model.eval()
torch.cuda.synchronize()
load_seconds = time.perf_counter() - load_start
rss_after = process.memory_info().rss / 1024**3
resolved_revision = getattr(model.config, "_commit_hash", None)

load_metrics = {
    "model_class": model.__class__.__name__,
    "resolved_revision": resolved_revision,
    "load_seconds": load_seconds,
    "process_rss_before_gib": rss_before,
    "process_rss_after_gib": rss_after,
    "gpu_allocated_gib": torch.cuda.memory_allocated() / 1024**3,
    "gpu_reserved_gib": torch.cuda.memory_reserved() / 1024**3,
    "gpu_peak_allocated_gib": torch.cuda.max_memory_allocated() / 1024**3,
    "parameter_count": sum(p.numel() for p in model.parameters()),
}
print(json.dumps(load_metrics, ensure_ascii=False, indent=2))


In [ ]:
# 셀 8: native chat template + greedy generation
def render_chat(prompt):
    if processor is not None:
        messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
        try:
            text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
            thinking_control = "enable_thinking_false"
        except TypeError:
            text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            thinking_control = "unsupported"
        batch = processor(text=[text], return_tensors="pt", add_special_tokens=False)
        decode = processor.decode
    else:
        messages = [{"role": "user", "content": prompt}]
        try:
            text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
            thinking_control = "enable_thinking_false"
        except TypeError:
            text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            thinking_control = "unsupported"
        batch = tokenizer(text, return_tensors="pt", add_special_tokens=False)
        decode = tokenizer.decode
    device = next(model.parameters()).device
    batch = {k: v.to(device) if hasattr(v, "to") else v for k, v in batch.items()}
    return batch, decode, thinking_control

@torch.inference_mode()
def generate_one(case):
    prompt = build_prompt(case)
    batch, decode, thinking_control = render_chat(prompt)
    input_tokens = int(batch["input_ids"].shape[-1])
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    output = model.generate(
        **batch,
        do_sample=False,
        max_new_tokens=MAX_NEW_TOKENS,
        pad_token_id=getattr(getattr(model, "generation_config", None), "pad_token_id", None)
            or getattr(getattr(model, "generation_config", None), "eos_token_id", None),
    )
    torch.cuda.synchronize()
    seconds = time.perf_counter() - start
    generated = output[0, input_tokens:]
    raw = decode(generated, skip_special_tokens=True).strip()
    return {
        "case_id": case["case_id"],
        "raw_output": raw,
        "input_tokens": input_tokens,
        "output_tokens": int(generated.numel()),
        "generation_seconds": seconds,
        "gpu_peak_allocated_gib": torch.cuda.max_memory_allocated() / 1024**3,
        "thinking_control": thinking_control,
    }

preview = generate_one(cases[0])
print(json.dumps(preview, ensure_ascii=False, indent=2))


In [ ]:
# 셀 9: 전체 선택 범위 실행. 사례마다 즉시 Drive에 저장하므로 중단돼도 앞 결과는 남습니다.
import datetime, statistics

run_name = f"{MODEL_KEY}_{RUN_SCOPE}_fp16_v3"
run_dir = os.path.join(DRIVE_OUTPUT_ROOT, run_name)
os.makedirs(run_dir, exist_ok=True)
pred_path = os.path.join(run_dir, "predictions.jsonl")

completed = set()
if os.path.exists(pred_path):
    with open(pred_path, encoding="utf-8") as f:
        completed = {json.loads(line)["case_id"] for line in f if line.strip()}
print("resume completed:", len(completed))

results = []
with open(pred_path, "a", encoding="utf-8") as f:
    for idx, case in enumerate(cases, 1):
        if case["case_id"] in completed:
            continue
        try:
            row = generate_one(case)
            row["status"] = "SUCCESS"
        except Exception as exc:
            row = {"case_id": case["case_id"], "raw_output": "", "status": "ERROR", "error_type": type(exc).__name__, "error": str(exc)}
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
        f.flush()
        results.append(row)
        print(f"[{idx}/{len(cases)}] {case['case_id']} {row['status']} {row.get('generation_seconds', 0):.3f}s")

with open(pred_path, encoding="utf-8") as f:
    saved_rows = [json.loads(line) for line in f if line.strip()]

manifest = {
    "experiment_id": "FP16_COMMON_V3",
    "run_name": run_name,
    "created_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "model_key": MODEL_KEY,
    "model_id": model_cfg["model_id"],
    "requested_revision": model_cfg["revision"],
    "resolved_revision": resolved_revision,
    "precision": "float16",
    "run_scope": RUN_SCOPE,
    "case_count": len(cases),
    "prompt_sha256": PROMPT_HASH,
    "max_new_tokens": MAX_NEW_TOKENS,
    "decoding": "greedy",
    "environment": versions,
    "load_metrics": load_metrics,
    "successful_cases": sum(r.get("status") == "SUCCESS" for r in saved_rows),
    "error_cases": sum(r.get("status") != "SUCCESS" for r in saved_rows),
    "mean_generation_seconds": statistics.mean([r["generation_seconds"] for r in saved_rows if "generation_seconds" in r]) if saved_rows else None,
    "max_gpu_peak_allocated_gib": max([r.get("gpu_peak_allocated_gib", 0) for r in saved_rows], default=None),
    "note": "Colab FP16 development measurement; not Android latency and not independent test accuracy",
}
with open(os.path.join(run_dir, "run_manifest.json"), "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)
print(json.dumps(manifest, ensure_ascii=False, indent=2))


In [ ]:
# 셀 10: 공통 채점기 실행
import subprocess, sys

if RUN_SCOPE == "gate":
    gate_gold = os.path.join(run_dir, "gate_gold.jsonl")
    with open(gate_gold, "w", encoding="utf-8") as f:
        for case in cases:
            f.write(json.dumps(case, ensure_ascii=False) + "\n")
    gold_args = ["--gold", gate_gold]
elif RUN_SCOPE == "selection":
    gold_args = ["--gold", core_path, "--gold", challenge_path]
else:
    gold_args = ["--gold", core_path, "--gold", challenge_path, "--gold", legacy_path]

score_dir = os.path.join(run_dir, "scores")
cmd = [
    sys.executable,
    os.path.join(EVAL_ROOT, "scripts", "score_predictions.py"),
    *gold_args,
    "--predictions", pred_path,
    "--out-dir", score_dir,
]
subprocess.run(cmd, check=True)


In [ ]:
# 셀 11: 모델 메모리 해제
del model
if processor is not None:
    del processor
if tokenizer is not None:
    del tokenizer
gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
print("모델 해제 완료, GPU allocated GiB:", torch.cuda.memory_allocated() / 1024**3)


In [ ]:
# 셀 12: Drive에 쌓인 selection 결과 비교표
import glob
import pandas as pd

rows = []
for summary_path in glob.glob(os.path.join(DRIVE_OUTPUT_ROOT, "*_selection_fp16_v3", "scores", "summary.json")):
    run_dir_found = str(pathlib.Path(summary_path).parents[1])
    with open(summary_path, encoding="utf-8") as f:
        summary = json.load(f)
    with open(os.path.join(run_dir_found, "run_manifest.json"), encoding="utf-8") as f:
        run = json.load(f)
    rows.append({
        "model_key": run["model_key"],
        "model_id": run["model_id"],
        "resolved_revision": run["resolved_revision"],
        "contract_valid_rate": summary["contract_valid_rate"],
        "exact_semantic_match_rate": summary["exact_semantic_match_rate"],
        "macro_intent_f1": summary["macro_intent_f1"],
        "fallback_exact_rate": summary["fallback_exact_rate"],
        "high_impact_exact_rate": summary["high_impact_exact_rate"],
        "dangerous_extra_action_cases": summary["dangerous_extra_action_cases"],
        "must_not_violation_cases": summary["must_not_violation_cases"],
        "load_seconds": run["load_metrics"]["load_seconds"],
        "mean_generation_seconds": run["mean_generation_seconds"],
        "max_gpu_peak_allocated_gib": run["max_gpu_peak_allocated_gib"],
    })

comparison = pd.DataFrame(rows)
if len(comparison):
    comparison = comparison.sort_values(
        ["dangerous_extra_action_cases", "exact_semantic_match_rate", "max_gpu_peak_allocated_gib"],
        ascending=[True, False, True],
    )
    display(comparison)
    comparison.to_csv(os.path.join(DRIVE_OUTPUT_ROOT, "fp16_selection_comparison.csv"), index=False, encoding="utf-8-sig")
else:
    print("아직 완료된 selection run이 없습니다.")
